In [126]:
from importlib import reload

import os
import random
import torch

import Data_Loader
reload(Data_Loader)
import NN_visualization
reload(NN_visualization)
import DonorFinder
reload(DonorFinder)
import RecipientFinder
reload(RecipientFinder)
import Graph_reconstruction
reload(Graph_reconstruction)

folder = "/mnt/c/Users/uhewm/Desktop/ProjectHGT/simulation_chunks"
#folder = "/mnt/c/ProjectHGT/simulation_chunks"

# ----------------------------------------------------------
# Load the NNs RecipientFinder and DonorFinder
# ----------------------------------------------------------

RecipientFinder_nn, RecipientFinder_threshold = NN_visualization.load_RecipientFinder(RecipientFinder.RecipientFinder)
DonorFinder_nn, global_max, global_min, eps = NN_visualization.load_DonorFinder(DonorFinder.DonorFinder)


Model loaded from /mnt/c/Users/uhewm/Desktop/ProjectHGT/recipient_finder_model.pt
Best threshold: 0.620
Model loaded from /mnt/c/Users/uhewm/Desktop/ProjectHGT/donorfinder_model.pt


In [224]:
import Data_Loader
reload(Data_Loader)
import NN_visualization
reload(NN_visualization)
import DonorFinder
reload(DonorFinder)
import RecipientFinder
reload(RecipientFinder)
import Graph_reconstruction
reload(Graph_reconstruction)

# ----------------------------------------------------------
# Load the chosen file
# ----------------------------------------------------------

files = [entry.path for entry in os.scandir(folder) if entry.is_file()]
file = random.choice(files)
graph = Data_Loader.load_file(file)

# ----------------------------------------------------------
# Find possible recipients
# ----------------------------------------------------------

RecipientFinder_input = graph.copy()
Data_Loader.aggregate_sequences(RecipientFinder_input)
RecipientFinder_input = Data_Loader.RecipientFinder_graph_to_dataset(RecipientFinder_input)

RecipientFinder_result = torch.sigmoid(RecipientFinder_nn(RecipientFinder_input.x, 
                                RecipientFinder_input.internal_node_data, 
                                RecipientFinder_input.level, 
                                RecipientFinder_input.edge_index))



recipient_node, recipient_node_prob = Graph_reconstruction.most_likely_recipient_node(G = data, scores = RecipientFinder_result, threshold = RecipientFinder_threshold)

print(recipient_node, recipient_node_prob)
if recipient_node is not None:
    print(graph.nodes[recipient_node]["recipient"])

# ----------------------------------------------------------
# Find possible recipients
# ----------------------------------------------------------

if recipient_node is not None:
    DonorFinder_input = graph.copy()
    Data_Loader.aggregate_sequences_fitch(DonorFinder_input)
    DonorFinder_input = Data_Loader.DonorFinder_graph_to_dataset(DonorFinder_input, recipient_node = recipient_node, p_false = 0)[0]

    gi = DonorFinder_input.graph_information - global_min
    gi = torch.log1p(gi)
    gi = gi / (global_max + eps)
    DonorFinder_input.graph_information = gi

    DonorFinder_result = DonorFinder_nn(
        DonorFinder_input.x,
        DonorFinder_input.internal_node_data,
        DonorFinder_input.graph_information.unsqueeze(0),
        DonorFinder_input.level,
        DonorFinder_input.edge_index,
        DonorFinder_input.valid_node,
        torch.zeros(DonorFinder_input.x.size(0), dtype=torch.long)  # single graph batch
    )

    valid_donor_mask = DonorFinder_input.possible_donor_mask.squeeze().bool()
    
    DonorFinder_result[~valid_donor_mask] = -1e9
    probs = torch.softmax(DonorFinder_result, dim=0)
    
    print(Graph_reconstruction.most_likely_donor_nodes(
        probs,
        valid_donor_mask,
        threshold=None
    ))

    Graph_reconstruction.visualize_graph_reconstruction(
        graph = graph,
        graph_tensor = DonorFinder_input,
        probs = probs,
        suspected_donor = Graph_reconstruction.most_likely_donor_nodes(
            probs,
            valid_donor_mask,
            threshold=None
        )[0][0]
    )



193 0.9755403399467468
{'is_parent_node': True, 'events': [((193, 30), (189, 170))]}
[(189, 0.551565408706665), (170, 0.13904856145381927), (163, 0.11791213601827621)]
198
/mnt/c/ProjectHGT/graph.html
>4;1H84l=                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

/mnt/c/Users/uhewm/OneDrive/PhD/Project No.2/pangenome-gene-transfer-simulation/Data_Loader.py:834: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  total_tree_length = torch.tensor(G.nodes[root]['tree_length'], dtype=torch.float32)


In [180]:
Graph_reconstruction.most_likely_donor_nodes(
        probs,
        valid_donor_mask,
        threshold=None
    )[0][0]

134